In [1]:
from pyspark.sql import functions as F
from data_generator import get_spark, generate_large_table
import time

spark=get_spark("Case02_Bucketing",{
    "spark.sql.sources.bucketing.enabled":"true",
    "spark.sql.autoBroadcastJoinThreshold": "-1"
})

events_df=generate_large_table(spark,num_rows=1000_000)
sessions_df=(
    generate_large_table(spark,num_rows=5_000)
    .withColumnRenamed("session_duration","total_session_time")
    .select("account_number","total_session_time","event_date")
)


26/04/11 17:18:53 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


# BAD: Join without bucketing (full shuffle)

In [2]:
events_df.write.mode("overwrite").parquet("/tmp/events_no_bucket")
sessions_df.write.mode("overwrite").parquet("/tmp/sessions_no_bucket")

26/04/11 17:18:57 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/11 17:18:57 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/04/11 17:18:57 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/04/11 17:18:57 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/04/11 17:18:57 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/04/11 17:18:57 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/04/11 17:18:57 WARN MemoryManager: Total allocation exceeds 95.

In [6]:
events_nb=spark.read.parquet("/tmp/events_no_bucket")
sessions_nb=spark.read.parquet("/tmp/sessions_no_bucket")
sessions_nb_clean = sessions_nb.drop("event_date")
start=time.time()
result_bad=events_nb.join(sessions_nb_clean,"account_number", "inner")
result_bad.write.mode("overwrite").parquet("/tmp/case02_bad")
print(f" Unbucketed join: {time.time() - start:.1f}s")

26/04/11 17:20:33 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/11 17:20:33 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/04/11 17:20:33 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/04/11 17:20:33 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/04/11 17:20:33 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/04/11 17:20:33 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/04/11 17:20:33 WARN MemoryManager: Total allocation exceeds 95.

 Unbucketed join: 2.6s


26/04/11 17:20:33 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/04/11 17:20:33 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/04/11 17:20:33 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/04/11 17:20:33 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/04/11 17:20:33 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/04/11 17:20:33 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


# GOOD: Write bucketed tables then join

In [8]:
NUM_BUCKETS = 64

(events_df
 .write
 .mode("overwrite")
 .bucketBy(NUM_BUCKETS, "account_number")
 .sortBy("account_number")
 .saveAsTable("events_bucketed"))

(sessions_df
 .write
 .mode("overwrite")
 .bucketBy(NUM_BUCKETS, "account_number")
 .sortBy("account_number")
 .saveAsTable("sessions_bucketed"))

events_b = spark.table("events_bucketed")
sessions_b = spark.table("sessions_bucketed")

start = time.time()
result_good = events_b.join(sessions_b, "account_number", "inner")
result_good.write.mode("overwrite").parquet("/tmp/case02_good")
print(f" Bucketed join: {time.time() - start:.1f}s")

Py4JJavaError: An error occurred while calling o183.saveAsTable.
: org.projectnessie.client.http.HttpClientException: Failed to execute GET request against 'http://nessie:19120/api/v1/trees/tree/main?fetch=MINIMAL'.
	at org.projectnessie.client.http.impl.jdk11.JavaRequest.executeRequest(JavaRequest.java:129)
	at org.projectnessie.client.http.HttpRequest.get(HttpRequest.java:106)
	at org.projectnessie.client.rest.v1.RestV1TreeClient.getReferenceByName(RestV1TreeClient.java:83)
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke(NativeMethodAccessorImpl.java:62)
	at java.base/jdk.internal.reflect.DelegatingMethodAccessorImpl.invoke(DelegatingMethodAccessorImpl.java:43)
	at java.base/java.lang.reflect.Method.invoke(Method.java:566)
	at org.projectnessie.client.rest.v1.RestV1Client$ExceptionRewriter.invoke(RestV1Client.java:84)
	at com.sun.proxy.$Proxy32.getReferenceByName(Unknown Source)
	at 